# Stage 3 — Raw Data Exploratory Data Analysis

This notebook provides assignment evidence for the generated `customers`, `orders`, and `tickets` CSV files. It documents shape, inferred types, blank values, value counts with percentages, duplicate patterns, inconsistent categorical labels, and domain outliers.

**Scope:** audit only. No raw values are changed, no cleaned files are created, and no SQLite database is loaded in Stage 3.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "generate_data.py").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.eda import (
    category_variants,
    column_profile,
    duplicate_summary,
    load_raw_data,
    quality_findings,
    shape_summary,
    value_count_summary,
)

datasets = load_raw_data(ROOT / "data" / "raw")
print("Loaded:", ", ".join(datasets))

Loaded: customers, orders, tickets


## 1. Dataset shapes and sample records

The customers and orders datasets exceed the 500-row requirement. Customers has exactly five columns, while orders has nine.

In [2]:
display(shape_summary(datasets))
for name, frame in datasets.items():
    print(f"\n{name}: {frame.shape[0]:,} rows × {frame.shape[1]} columns")
    display(frame.head(3))

,dataset,rows,columns
0,customers,1200,5
1,orders,13084,9
2,tickets,164,5



customers: 1,200 rows × 5 columns


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,MALE
1,C0002,DELHI,2024-02-17,31,MALE
2,C0003,bengaluru,2025-02-18,18,F



orders: 13,084 rows × 9 columns


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
0,ORD-108250,C0792,P004,2,2025-09-01 12:24:59,completed,0,310.0,17
1,ORD-101939,C0190,P003,1,2025-08-17 07:07:23,completed,5,232.75,21
2,ORD-108765,C0841,P033,1,2025-07-13 09:42:32,completed,10,1349.1,27



tickets: 164 rows × 5 columns


,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical


## 2. Data types, completeness, and cardinality

The loader uses `keep_default_na=False`, preserving intentional blank strings for an honest raw-data audit. `blank_count` therefore includes both true missing values and empty/whitespace-only strings.

In [3]:
profiles = column_profile(datasets)
display(profiles)

,dataset,column,dtype,blank_count,blank_pct,unique_non_blank
0,customers,customer_id,object,0,0.00,1200
1,customers,city,object,0,0.00,24
2,customers,signup_date,object,0,0.00,432
3,customers,age,object,61,5.08,49
4,customers,gender,object,30,2.50,10
5,orders,order_id,object,0,0.00,12309
6,orders,customer_id,object,0,0.00,1182
7,orders,product_id,object,0,0.00,40
8,orders,quantity,int64,0,0.00,3
9,orders,order_ts,object,0,0.00,12959


## 3. Column value counts and percentages

All low-cardinality values are shown. For high-cardinality columns, the ten most frequent values are shown and the remainder is grouped into an `<OTHER>` row so counts still reconcile to the dataset total.

In [4]:
value_counts = value_count_summary(datasets)
for name in datasets:
    print(f"\n{name}")
    display(value_counts[value_counts["dataset"] == name].reset_index(drop=True))


customers


,dataset,column,value,count,percentage
0,customers,customer_id,C0001,1,0.08
1,customers,customer_id,C0807,1,0.08
2,customers,customer_id,C0805,1,0.08
3,customers,customer_id,C0804,1,0.08
4,customers,customer_id,C0803,1,0.08
5,customers,customer_id,C0802,1,0.08
6,customers,customer_id,C0801,1,0.08
7,customers,customer_id,C0800,1,0.08
8,customers,customer_id,C0799,1,0.08
9,customers,customer_id,C0798,1,0.08



orders


,dataset,column,value,count,percentage
0,orders,order_id,ORD-108807,3,0.02
1,orders,order_id,ORD-111253,3,0.02
2,orders,order_id,ORD-100107,3,0.02
3,orders,order_id,ORD-104391,2,0.02
4,orders,order_id,ORD-102965,2,0.02
...,...,...,...,...,...
74,orders,delivery_minutes,23,454,3.47
75,orders,delivery_minutes,27,450,3.44
76,orders,delivery_minutes,20,420,3.21
77,orders,delivery_minutes,22,415,3.17



tickets


,dataset,column,value,count,percentage
0,tickets,ticket_id,T0024,1,0.61
1,tickets,ticket_id,T0025,1,0.61
2,tickets,ticket_id,T0148,1,0.61
3,tickets,ticket_id,T0088,1,0.61
4,tickets,ticket_id,T0030,1,0.61
5,tickets,ticket_id,T0067,1,0.61
6,tickets,ticket_id,T0029,1,0.61
7,tickets,ticket_id,T0129,1,0.61
8,tickets,ticket_id,T0105,1,0.61
9,tickets,ticket_id,T0045,1,0.61


## 4. Duplicate analysis

Exact duplicate rows are different from repeated order IDs. Repeated order IDs can also represent an earlier `pending` state followed by a final state; Stage 3 identifies these patterns but does not resolve them.

In [5]:
display(duplicate_summary(datasets))

,dataset,key,exact_duplicate_rows,unique_keys,duplicate_key_excess_rows,rows_with_repeated_key
0,customers,customer_id,0,1200,0,0
1,orders,order_id,119,12309,775,1547
2,tickets,ticket_id,0,164,0,0


## 5. Inconsistent labels, blanks, and outlier flags

These are diagnostic counts based on explicit rules. For example, the instructor audit rule flags age below 10 or above 100, while delivery time cannot be negative.

In [6]:
display(quality_findings(datasets))

,dataset,column,finding,count,interpretation
0,customers,city,raw spelling variants,24,Whitespace and letter case produce 24 raw valu...
1,customers,city,normalized candidate values,6,A strip-and-title preview yields six cities; n...
2,customers,gender,nonblank spelling variants,10,Male and female labels appear with inconsisten...
3,customers,gender,blank values,30,Blank values require an explicit Unknown mappi...
4,customers,age,blank or nonnumeric values,61,Blank ages are preserved during EDA.
5,customers,age,domain outliers,6,Ages below 10 or above 100 violate the instruc...
6,orders,order_id,exact duplicate rows,119,These rows are identical across every column.
7,orders,order_id,duplicate-key excess rows,775,Repeated IDs include exact duplicates and earl...
8,orders,status,pending rows,656,Pending records may be earlier states of a fin...
9,orders,order_amount,nonnumeric values,120,The raw amount column contains N/A and error s...


In [7]:
variants = category_variants(datasets)
for column in ("city", "gender"):
    print(f"\nRaw {column} labels and normalization preview")
    display(variants[variants["column"] == column].reset_index(drop=True))


Raw city labels and normalization preview


,column,raw_value,normalized_preview,count,percentage
0,city,Pune,Pune,148,12.33
1,city,Delhi,Delhi,123,10.25
2,city,Hyderabad,Hyderabad,119,9.92
3,city,Bengaluru,Bengaluru,115,9.58
4,city,Mumbai,Mumbai,113,9.42
5,city,Chennai,Chennai,106,8.83
6,city,chennai,Chennai,44,3.67
7,city,hyderabad,Hyderabad,40,3.33
8,city,delhi,Delhi,36,3.00
9,city,pune,Pune,35,2.92



Raw gender labels and normalization preview


,column,raw_value,normalized_preview,count,percentage
0,gender,female,Female,136,11.33
1,gender,FEMALE,Female,122,10.17
2,gender,F,Female,120,10.00
3,gender,m,Male,118,9.83
4,gender,f,Female,117,9.75
5,gender,Female,Female,115,9.58
6,gender,male,Male,114,9.50
7,gender,MALE,Male,110,9.17
8,gender,M,Male,110,9.17
9,gender,Male,Male,108,9.00


## 6. Stage 3 conclusions

- Customers contains 1,200 rows and five columns, satisfying the minimum dataset requirement. City and gender are categorical columns with deliberate spelling/case inconsistencies.
- Orders contains 13,084 raw rows but 12,309 distinct order IDs. Exact duplicates and earlier order states must be handled separately.
- Numeric-looking fields include invalid strings, blanks, and impossible values, so explicit coercion rules are needed before analysis.
- Tickets includes blank and overlong text that will need guardrails before later AI processing.
- The next stage should define and test cleaning rules, write trusted outputs, load SQLite, and compare Pandas results with SQL queries.